In [1]:
import sys
import importlib
sys.path.append("/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/")
import python.utils as ut

import os
import numpy as np
import arviz as az
from numpy.polynomial.legendre import legvander
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
import json
import glob
from scipy.stats import norm, cauchy, mode, t
from cmdstanpy import from_csv
import seaborn as sns
from tqdm import tqdm
import corner
import json

plt.style.use('seaborn-v0_8')

In [2]:
L_desired = 8
prior_path = "/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/data/json/priors.json"
data_path = "/users/jmduchar/data/jmduchar/Research/mcgill25/rfi_characterization/data/json/legendre_supervised.json"

In [3]:
def transition_counts(s, starts):
    C = np.zeros((4,4), dtype=float)
    for t in range(1, len(s)):
        if t in starts: 
            continue
        i, j = s[t-1]-1, s[t]-1
        C[i, j] += 1
    return C

def lognorm_params_from_samples(x, temper=2.0):
    z = np.log(x)
    mu, sd = z.mean(), z.std(ddof=1)
    return mu, sd/temper

def beta_params_from_samples(x, kappa=0.5, eps=0.1):
    m, v = x.mean(), x.var(ddof=1)
    
    # Guard against tiny variance:
    v = max(v, 1e-6 * m*(1-m))
    a = m*(m*(1-m)/v - 1.0)
    b = (1-m)*(m*(1-m)/v - 1.0)
    return a*kappa + eps, b*kappa + eps

In [4]:
outputs = glob.glob("../stan/stan_out/legendre-20251029143740_*.csv")

fit = from_csv(outputs)

14:20:32 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 1000 iterations at max treedepth (100.0%)
	Chain 2 had 1000 iterations at max treedepth (100.0%)
	Chain 3 had 1000 iterations at max treedepth (100.0%)
	Chain 4 had 1000 iterations at max treedepth (100.0%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [5]:
# Load JSON data
with open(
    data_path,
    "r"
) as f:
    data_dict = json.load(f)
    
post = fit.stan_variables()
start_idx = data_dict['start_idx_sup']
s = data_dict['s_sup']

In [6]:
C = transition_counts(s, set(start_idx))

# tampering
lam = 0.5

# weak prior
eta_clean  = np.array([9.0, 1.0, 0.2])
eta_rising = np.array([8.0, 2.0, 0.2])
eta_decay  = np.array([5.0, 0.5, 4.5, 0.2])
eta_blip   = np.array([8.0, 0.5, 0.5, 0.1])

alpha_clean  = eta_clean  + lam * C[0, [0,1,3]]
alpha_rising = eta_rising + lam * C[1, [1,2,3]]
alpha_decay  = eta_decay  + lam * C[2, [0,1,2,3]]
alpha_blip   = eta_blip   + lam * C[3, [0,1,2,3]]

rr_log_mu, rr_log_sigma = lognorm_params_from_samples(post["rate_rising"], temper=2.0)
# sig_log_mu, sig_log_sigma = lognorm_params_from_samples(post["sigma"], temper=2.0)
sig_log_mu, sig_log_sigma = -1.11, 0.5
k_log_mu,  k_log_sigma  = lognorm_params_from_samples(post["k_blip"], temper=2.0)

rd_alpha, rd_beta = beta_params_from_samples(post["rate_decay"], kappa=0.5, eps=0.1)

mu_blip_mean = float(np.mean(post["mu_blip"]))
mu_blip_sd   = float(np.std(post["mu_blip"], ddof=1) * 2.0)

In [7]:
L_previous = len(post["mu_X"].mean(axis=0))

if L_desired == L_previous:
    print("Initializing Legendre coefficient priors based on previous run.")
    mu_X_mean = post["mu_X"].mean(axis=0)
    mu_X_sd   = post["mu_X"].std(axis=0, ddof=1) * 2.0
    alpha_X_log_mu, alpha_X_log_sigma = np.log(post["alpha_X"]).mean(axis=0), np.log(post["alpha_X"]).std(axis=0, ddof=1)/2
    beta_X_log_mu, beta_X_log_sigma   = np.log(post["beta_X"]).mean(), max(np.log(post["beta_X"]).std(ddof=1)/2, 0.1)
    
else:
    print("Initializing Legendre coefficient priors from scratch.")
    
    mu_X_mean = np.full(L_desired, -1.0, dtype=float)
    mu_X_sd   = np.full(L_desired, 3.0 * 2.0, dtype=float)

    alpha_X_log_mu    = np.full(L_desired, np.log(70.0), dtype=float)
    alpha_X_log_sigma = np.full(L_desired, 100.0, dtype=float)

    beta_X_log_mu    = float(np.log(2.0))
    beta_X_log_sigma = float(0.35)

Initializing Legendre coefficient priors from scratch.


In [9]:
prior_dict = dict(
    alpha_clean       = alpha_clean.tolist(), 
    alpha_rising      = alpha_rising.tolist(),
    alpha_decay       = alpha_decay.tolist(), 
    alpha_blip        = alpha_blip.tolist(),
    rr_log_mu         = rr_log_mu.tolist(), 
    rr_log_sigma      = rr_log_sigma.tolist(),
    rd_alpha          = rd_alpha.tolist(), 
    rd_beta           = rd_beta.tolist(),
    sig_log_mu        = sig_log_mu, 
    sig_log_sigma     = sig_log_sigma,
    mu_blip_mean      = mu_blip_mean, 
    mu_blip_sd        = mu_blip_sd,
    k_blip_log_mu     = k_log_mu.tolist(), 
    k_blip_log_sigma  = k_log_sigma.tolist(),
    mu_X_mean         = mu_X_mean.tolist(), 
    mu_X_sd           = mu_X_sd.tolist(),
    alpha_X_log_mu    = alpha_X_log_mu.tolist(), 
    alpha_X_log_sigma = alpha_X_log_sigma.tolist(),
    beta_X_log_mu     = beta_X_log_mu, 
    beta_X_log_sigma  = beta_X_log_sigma,
#     beta_X_log_mu     = beta_X_log_mu.tolist(), 
#     beta_X_log_sigma  = beta_X_log_sigma.tolist(),
)

In [10]:
with open(
    prior_path,
    "w"
) as f:
    json.dump(prior_dict, f, indent=2)